# Notebook 05 — Baseline Machine Learning Models and Internal Validation

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI Data  
**Notebook role:** Baseline model development and internal validation  
**Primary outcome:** Rapid motor progression at the recommended follow-up window from Notebook 02  
**Target variable:** `rapid_progression_q75`  
**Important:** This notebook uses the preprocessed train/test matrices generated by Notebook 04. It does not re-fit preprocessing on the test set.

## Objective

Develop and internally validate baseline machine learning models for predicting rapid Parkinson’s disease motor progression using baseline predictors only.

This notebook will:

1. Load the train/test split and preprocessed matrices generated by Notebook 04.
2. Verify target distribution and feature integrity.
3. Train baseline classification models using the training set only.
4. Perform stratified cross-validation within the training set.
5. Select operating thresholds from training cross-validation only.
6. Evaluate final models on the held-out test set.
7. Run a sensitivity analysis excluding `baseline_NP3TOT`.
8. Save metrics, predictions, model ranking, feature-importance summaries, plots, and a quality-control checklist.

## Scientific Background

Parkinson’s disease progression is heterogeneous. A reproducible prediction model must therefore use baseline information only and must avoid data leakage from future visits or outcome-derived variables.

The current target is a binary progression label created in Notebook 02:

- `rapid_progression_q75 = 1`: participant is in the upper quartile of annualized MDS-UPDRS Part III worsening.
- `rapid_progression_q75 = 0`: participant is below that threshold.

Because rapid progressors represent approximately one quarter of the analytic cohort, accuracy alone is not sufficient. This notebook reports discrimination and clinically relevant classification metrics, including ROC-AUC, PR-AUC, balanced accuracy, sensitivity, specificity, precision, and F1 score.

## Dataset Verification

This notebook expects the following outputs from Notebook 04:

```text
MyDrive/PPMI_PD_Progression/outputs/notebook_04_preprocessing/
├── 04_train_raw_split_before_preprocessing.csv
├── 05_test_raw_split_before_preprocessing.csv
├── 06_train_processed_matrix.csv
├── 07_test_processed_matrix.csv
├── 08_processed_feature_names.csv
├── 09_fitted_preprocessing_pipeline.joblib
├── 10_sensitivity_feature_set_plan.csv
└── 11_processed_feature_names_without_baseline_NP3TOT.csv
```

The raw PPMI participant-level data should remain private and should not be uploaded to public repositories.

## Expected Output

The notebook will save results to:

```text
MyDrive/PPMI_PD_Progression/outputs/notebook_05_baseline_ml/
```

Main output files:

```text
01_dataset_verification.csv
02_train_target_distribution.csv
03_cv_performance_primary_models.csv
04_test_performance_primary_models.csv
05_model_ranking.csv
06_test_predictions_all_models.csv
07_selected_primary_model.txt
08_sensitivity_cv_performance_no_baseline_NP3TOT.csv
09_sensitivity_test_performance_no_baseline_NP3TOT.csv
10_feature_importance_best_model_training_only.csv
11_quality_control_checklist.csv
12_notebook_05_summary_report.txt
fig_roc_curves_test.png
fig_pr_curves_test.png
fig_calibration_best_model_test.png
```

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix, brier_score_loss,
    roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
TARGET_COL = "rapid_progression_q75"
ID_COL = "PATNO"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")
NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_05_baseline_ml"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Notebook 04 output directory:", NB04_DIR)
print("Notebook 05 output directory:", OUT_DIR)
print("Notebook 04 directory exists:", NB04_DIR.exists())

In [ ]:
# ============================================================
# 03. Required file verification
# ============================================================

required_files = {
    "train_raw": NB04_DIR / "04_train_raw_split_before_preprocessing.csv",
    "test_raw": NB04_DIR / "05_test_raw_split_before_preprocessing.csv",
    "train_processed": NB04_DIR / "06_train_processed_matrix.csv",
    "test_processed": NB04_DIR / "07_test_processed_matrix.csv",
    "processed_feature_names": NB04_DIR / "08_processed_feature_names.csv",
    "preprocessing_pipeline": NB04_DIR / "09_fitted_preprocessing_pipeline.joblib",
    "sensitivity_plan": NB04_DIR / "10_sensitivity_feature_set_plan.csv",
    "processed_feature_names_no_baseline_NP3TOT": NB04_DIR / "11_processed_feature_names_without_baseline_NP3TOT.csv",
}

verification_rows = []
for key, path in required_files.items():
    verification_rows.append({
        "file_key": key,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan
    })

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(OUT_DIR / "01_dataset_verification.csv", index=False)
display(verification_df)

missing = verification_df.loc[~verification_df["exists"], "file_key"].tolist()
if missing:
    raise FileNotFoundError(
        "Missing required Notebook 04 output files: " + ", ".join(missing) +
        "\nPlease run Notebook 04 completely before running Notebook 05."
    )

In [ ]:
# ============================================================
# 04. Helper functions
# ============================================================

def remove_unnamed_index_columns(df):
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed:")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df

def read_csv_clean(path):
    df = pd.read_csv(path)
    df = remove_unnamed_index_columns(df)
    return df

def read_feature_names(path):
    df = pd.read_csv(path)
    df = remove_unnamed_index_columns(df)
    if df.shape[1] == 1:
        names = df.iloc[:, 0].dropna().astype(str).tolist()
    else:
        # Prefer a named feature column if available
        possible_cols = [c for c in df.columns if "feature" in c.lower()]
        if possible_cols:
            names = df[possible_cols[0]].dropna().astype(str).tolist()
        else:
            names = df.iloc[:, 0].dropna().astype(str).tolist()
    return names

def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def safe_metric(metric_func, y_true, y_pred_or_prob, **kwargs):
    try:
        return metric_func(y_true, y_pred_or_prob, **kwargs)
    except Exception:
        return np.nan

def choose_threshold_youden(y_true, proba):
    fpr, tpr, thresholds = roc_curve(y_true, proba)
    j = tpr - fpr
    best_idx = int(np.nanargmax(j))
    thr = thresholds[best_idx]
    # Protect against infinite thresholds occasionally returned by roc_curve
    if not np.isfinite(thr):
        finite_thresholds = thresholds[np.isfinite(thresholds)]
        thr = float(np.max(finite_thresholds)) if len(finite_thresholds) else 0.5
    return float(thr)

def evaluate_predictions(y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "roc_auc": safe_metric(roc_auc_score, y_true, proba),
        "pr_auc_average_precision": safe_metric(average_precision_score, y_true, proba),
        "brier_score": safe_metric(brier_score_loss, y_true, proba),
        "accuracy": safe_metric(accuracy_score, y_true, y_pred),
        "balanced_accuracy": safe_metric(balanced_accuracy_score, y_true, y_pred),
        "sensitivity_recall": safe_metric(recall_score, y_true, y_pred, zero_division=0),
        "specificity": specificity_score(y_true, y_pred),
        "precision": safe_metric(precision_score, y_true, y_pred, zero_division=0),
        "f1": safe_metric(f1_score, y_true, y_pred, zero_division=0),
        "positive_predictions": int(np.sum(y_pred)),
        "negative_predictions": int(len(y_pred) - np.sum(y_pred)),
    }

def clone_model(model):
    # sklearn clone avoids carrying fitted state
    from sklearn.base import clone
    return clone(model)

In [ ]:
# ============================================================
# 05. Load train/test matrices and target
# ============================================================

train_raw = read_csv_clean(required_files["train_raw"])
test_raw = read_csv_clean(required_files["test_raw"])
train_processed = read_csv_clean(required_files["train_processed"])
test_processed = read_csv_clean(required_files["test_processed"])

processed_feature_names = read_feature_names(required_files["processed_feature_names"])
sensitivity_feature_names = read_feature_names(required_files["processed_feature_names_no_baseline_NP3TOT"])

print("train_raw shape:", train_raw.shape)
print("test_raw shape:", test_raw.shape)
print("train_processed shape:", train_processed.shape)
print("test_processed shape:", test_processed.shape)
print("Number of processed feature names:", len(processed_feature_names))
print("Number of sensitivity feature names:", len(sensitivity_feature_names))

if TARGET_COL in train_raw.columns and TARGET_COL in test_raw.columns:
    y_train = train_raw[TARGET_COL].astype(int).reset_index(drop=True)
    y_test = test_raw[TARGET_COL].astype(int).reset_index(drop=True)
elif TARGET_COL in train_processed.columns and TARGET_COL in test_processed.columns:
    y_train = train_processed[TARGET_COL].astype(int).reset_index(drop=True)
    y_test = test_processed[TARGET_COL].astype(int).reset_index(drop=True)
else:
    raise ValueError(f"Target column '{TARGET_COL}' was not found in raw or processed split files.")

# Select processed feature columns.
available_feature_names = [c for c in processed_feature_names if c in train_processed.columns and c in test_processed.columns]
missing_feature_names = sorted(set(processed_feature_names) - set(available_feature_names))

if missing_feature_names:
    print("Warning: some listed feature names were not found in the processed matrices:")
    print(missing_feature_names[:20], "..." if len(missing_feature_names) > 20 else "")

if len(available_feature_names) == 0:
    # Fallback: use all columns except ID and target
    excluded = {ID_COL, TARGET_COL}
    available_feature_names = [c for c in train_processed.columns if c not in excluded]
    print("Fallback feature selection was used.")

X_train = train_processed[available_feature_names].copy().reset_index(drop=True)
X_test = test_processed[available_feature_names].copy().reset_index(drop=True)

assert len(X_train) == len(y_train), "X_train and y_train row counts do not match."
assert len(X_test) == len(y_test), "X_test and y_test row counts do not match."

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts().sort_index())
print("y_test distribution:")
print(y_test.value_counts().sort_index())

In [ ]:
# ============================================================
# 06. Target distribution and basic integrity checks
# ============================================================

target_distribution = pd.DataFrame([
    {"set": "train", "n": len(y_train), "positive_n": int(y_train.sum()), "negative_n": int((1-y_train).sum()), "positive_percent": float(y_train.mean()*100)},
    {"set": "test", "n": len(y_test), "positive_n": int(y_test.sum()), "negative_n": int((1-y_test).sum()), "positive_percent": float(y_test.mean()*100)},
    {"set": "overall", "n": len(y_train)+len(y_test), "positive_n": int(y_train.sum()+y_test.sum()), "negative_n": int((1-y_train).sum()+(1-y_test).sum()), "positive_percent": float(pd.concat([y_train,y_test]).mean()*100)},
])
target_distribution.to_csv(OUT_DIR / "02_train_target_distribution.csv", index=False)
display(target_distribution)

print("Missing values in X_train:", int(X_train.isna().sum().sum()))
print("Missing values in X_test:", int(X_test.isna().sum().sum()))

if X_train.isna().sum().sum() > 0 or X_test.isna().sum().sum() > 0:
    raise ValueError("Processed matrices contain missing values. Re-check Notebook 04 preprocessing.")

# Optional ID order checks
if ID_COL in train_raw.columns and ID_COL in train_processed.columns:
    same_train_ids = train_raw[ID_COL].reset_index(drop=True).equals(train_processed[ID_COL].reset_index(drop=True))
    print("Train PATNO order matches between raw and processed:", same_train_ids)
if ID_COL in test_raw.columns and ID_COL in test_processed.columns:
    same_test_ids = test_raw[ID_COL].reset_index(drop=True).equals(test_processed[ID_COL].reset_index(drop=True))
    print("Test PATNO order matches between raw and processed:", same_test_ids)

## Code — Model Training Plan

Models are intentionally simple and reproducible:

1. **Dummy classifier** — reference baseline.
2. **Logistic regression with class weighting** — interpretable linear baseline.
3. **Logistic regression without class weighting** — comparison baseline.
4. **Random forest with balanced subsampling** — nonlinear tree-based model.
5. **Gradient boosting** — nonlinear boosting baseline.
6. **Histogram gradient boosting** — scalable boosting baseline.

Model ranking is based on cross-validation performance within the training set, not on test-set performance.

In [ ]:
# ============================================================
# 07. Define baseline models
# ============================================================

models = {
    "Dummy_Most_Frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic_Regression_Balanced": LogisticRegression(
        max_iter=5000, class_weight="balanced", solver="liblinear", random_state=RANDOM_STATE
    ),
    "Logistic_Regression_L2": LogisticRegression(
        max_iter=5000, class_weight=None, solver="liblinear", random_state=RANDOM_STATE
    ),
    "Random_Forest_Balanced": RandomForestClassifier(
        n_estimators=500, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Hist_Gradient_Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Models:")
for name in models:
    print("-", name)

In [ ]:
# ============================================================
# 08. Cross-validation on training set
# ============================================================

cv_rows = []
cv_thresholds = {}
oof_probabilities = {}

for name, model in models.items():
    print(f"Running CV for: {name}")
    estimator = clone_model(model)

    # OOF probability from training folds only
    try:
        oof_proba = cross_val_predict(
            estimator, X_train, y_train, cv=cv, method="predict_proba", n_jobs=None
        )[:, 1]
    except Exception as e:
        print(f"Model failed during CV: {name} -> {e}")
        continue

    oof_probabilities[name] = oof_proba

    # Default threshold
    metrics_05 = evaluate_predictions(y_train.values, oof_proba, threshold=0.5)
    metrics_05.update({"model": name, "evaluation": "cv_oof_threshold_0.5"})
    cv_rows.append(metrics_05)

    # Youden threshold from OOF training probabilities only
    thr = choose_threshold_youden(y_train.values, oof_proba)
    cv_thresholds[name] = thr
    metrics_youden = evaluate_predictions(y_train.values, oof_proba, threshold=thr)
    metrics_youden.update({"model": name, "evaluation": "cv_oof_training_derived_youden_threshold"})
    cv_rows.append(metrics_youden)

cv_perf = pd.DataFrame(cv_rows)
cols = ["model", "evaluation"] + [c for c in cv_perf.columns if c not in ["model", "evaluation"]]
cv_perf = cv_perf[cols]
cv_perf.to_csv(OUT_DIR / "03_cv_performance_primary_models.csv", index=False)
display(cv_perf.sort_values(["evaluation", "roc_auc"], ascending=[True, False]))

In [ ]:
# ============================================================
# 09. Fit models on full training set and evaluate on held-out test set
# ============================================================

test_rows = []
test_predictions = pd.DataFrame()

# Add IDs if available
if ID_COL in test_raw.columns:
    test_predictions[ID_COL] = test_raw[ID_COL].values
test_predictions["y_true"] = y_test.values

fitted_models = {}

for name, model in models.items():
    if name not in cv_thresholds:
        continue

    print(f"Fitting final model: {name}")
    estimator = clone_model(model)
    estimator.fit(X_train, y_train)
    fitted_models[name] = estimator

    test_proba = estimator.predict_proba(X_test)[:, 1]
    test_predictions[f"proba_{name}"] = test_proba

    # Test with default threshold
    m05 = evaluate_predictions(y_test.values, test_proba, threshold=0.5)
    m05.update({"model": name, "evaluation": "test_threshold_0.5"})
    test_rows.append(m05)

    # Test with training-derived threshold
    thr = cv_thresholds[name]
    my = evaluate_predictions(y_test.values, test_proba, threshold=thr)
    my.update({"model": name, "evaluation": "test_training_derived_youden_threshold"})
    test_rows.append(my)

test_perf = pd.DataFrame(test_rows)
cols = ["model", "evaluation"] + [c for c in test_perf.columns if c not in ["model", "evaluation"]]
test_perf = test_perf[cols]
test_perf.to_csv(OUT_DIR / "04_test_performance_primary_models.csv", index=False)
test_predictions.to_csv(OUT_DIR / "06_test_predictions_all_models.csv", index=False)

display(test_perf.sort_values(["evaluation", "roc_auc"], ascending=[True, False]))
display(test_predictions.head())

In [ ]:
# ============================================================
# 10. Model ranking based on training cross-validation
# ============================================================

ranking_base = cv_perf[cv_perf["evaluation"] == "cv_oof_training_derived_youden_threshold"].copy()

# Exclude dummy from selecting the primary model unless all models failed
non_dummy = ranking_base[~ranking_base["model"].str.contains("Dummy", case=False, na=False)].copy()
ranking_source = non_dummy if len(non_dummy) > 0 else ranking_base

ranking = ranking_source.sort_values(
    by=["roc_auc", "pr_auc_average_precision", "balanced_accuracy"],
    ascending=[False, False, False]
).reset_index(drop=True)

ranking["rank"] = np.arange(1, len(ranking) + 1)
ranking = ranking[["rank"] + [c for c in ranking.columns if c != "rank"]]
ranking.to_csv(OUT_DIR / "05_model_ranking.csv", index=False)

selected_model_name = ranking.loc[0, "model"]
selected_threshold = float(cv_thresholds[selected_model_name])

with open(OUT_DIR / "07_selected_primary_model.txt", "w") as f:
    f.write(f"Selected primary model: {selected_model_name}\n")
    f.write(f"Selection rule: highest training CV ROC-AUC, then PR-AUC, then balanced accuracy.\n")
    f.write(f"Selected threshold: {selected_threshold:.6f}\n")
    f.write(f"Threshold derivation: Youden index using out-of-fold training predictions only.\n")

display(ranking)
print("Selected primary model:", selected_model_name)
print("Selected threshold:", selected_threshold)

## Scientific Interpretation Checkpoint

At this stage, the selected model is still an internally validated baseline model. It should not be interpreted as a final clinical prediction tool until additional validation, calibration assessment, sensitivity analysis, and external validation are performed.

In [ ]:
# ============================================================
# 11. Sensitivity analysis excluding baseline_NP3TOT
# ============================================================

available_sensitivity_features = [
    c for c in sensitivity_feature_names if c in train_processed.columns and c in test_processed.columns
]

if len(available_sensitivity_features) == 0:
    print("No sensitivity feature names found. Sensitivity analysis will be skipped.")
    sens_cv_perf = pd.DataFrame()
    sens_test_perf = pd.DataFrame()
else:
    print("Sensitivity features:", len(available_sensitivity_features))
    X_train_sens = train_processed[available_sensitivity_features].copy().reset_index(drop=True)
    X_test_sens = test_processed[available_sensitivity_features].copy().reset_index(drop=True)

    sens_cv_rows = []
    sens_thresholds = {}
    sens_test_rows = []

    for name, model in models.items():
        print(f"Sensitivity CV: {name}")
        estimator = clone_model(model)
        try:
            oof_proba = cross_val_predict(
                estimator, X_train_sens, y_train, cv=cv, method="predict_proba", n_jobs=None
            )[:, 1]
        except Exception as e:
            print(f"Sensitivity model failed during CV: {name} -> {e}")
            continue

        thr = choose_threshold_youden(y_train.values, oof_proba)
        sens_thresholds[name] = thr

        mcv = evaluate_predictions(y_train.values, oof_proba, threshold=thr)
        mcv.update({"model": name, "evaluation": "sensitivity_cv_no_baseline_NP3TOT_training_derived_youden_threshold"})
        sens_cv_rows.append(mcv)

        # Fit and evaluate on test
        estimator_final = clone_model(model)
        estimator_final.fit(X_train_sens, y_train)
        test_proba = estimator_final.predict_proba(X_test_sens)[:, 1]
        mt = evaluate_predictions(y_test.values, test_proba, threshold=thr)
        mt.update({"model": name, "evaluation": "sensitivity_test_no_baseline_NP3TOT_training_derived_youden_threshold"})
        sens_test_rows.append(mt)

    sens_cv_perf = pd.DataFrame(sens_cv_rows)
    sens_test_perf = pd.DataFrame(sens_test_rows)

sens_cv_perf.to_csv(OUT_DIR / "08_sensitivity_cv_performance_no_baseline_NP3TOT.csv", index=False)
sens_test_perf.to_csv(OUT_DIR / "09_sensitivity_test_performance_no_baseline_NP3TOT.csv", index=False)

display(sens_cv_perf.sort_values("roc_auc", ascending=False) if not sens_cv_perf.empty else sens_cv_perf)
display(sens_test_perf.sort_values("roc_auc", ascending=False) if not sens_test_perf.empty else sens_test_perf)

In [ ]:
# ============================================================
# 12. Feature importance for selected model using training data only
# ============================================================

best_model = fitted_models[selected_model_name]

importance_df = pd.DataFrame()

# Try model-native importance first
if hasattr(best_model, "coef_"):
    coefs = best_model.coef_[0]
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance_value": coefs,
        "abs_importance": np.abs(coefs),
        "importance_type": "logistic_regression_coefficient"
    }).sort_values("abs_importance", ascending=False)
elif hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance_value": importances,
        "abs_importance": np.abs(importances),
        "importance_type": "model_native_feature_importance"
    }).sort_values("abs_importance", ascending=False)

# If native importance is unavailable, use training-set permutation importance.
if importance_df.empty:
    try:
        perm = permutation_importance(
            best_model, X_train, y_train,
            n_repeats=10, random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
        )
        importance_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance_value": perm.importances_mean,
            "abs_importance": np.abs(perm.importances_mean),
            "importance_type": "training_permutation_importance_roc_auc"
        }).sort_values("abs_importance", ascending=False)
    except Exception as e:
        print("Could not compute feature importance:", e)

importance_df.to_csv(OUT_DIR / "10_feature_importance_best_model_training_only.csv", index=False)
display(importance_df.head(20))

In [ ]:
# ============================================================
# 13. Save ROC, PR, and calibration plots
# ============================================================

# ROC curves
plt.figure(figsize=(7, 6))
for name in fitted_models:
    proba_col = f"proba_{name}"
    if proba_col in test_predictions.columns:
        fpr, tpr, _ = roc_curve(y_test, test_predictions[proba_col])
        auc = roc_auc_score(y_test, test_predictions[proba_col])
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_roc_curves_test.png", dpi=300)
plt.show()

# Precision-recall curves
plt.figure(figsize=(7, 6))
for name in fitted_models:
    proba_col = f"proba_{name}"
    if proba_col in test_predictions.columns:
        precision, recall, _ = precision_recall_curve(y_test, test_predictions[proba_col])
        ap = average_precision_score(y_test, test_predictions[proba_col])
        plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Precision-Recall Curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_pr_curves_test.png", dpi=300)
plt.show()

# Calibration curve for selected model
selected_proba = test_predictions[f"proba_{selected_model_name}"].values
frac_pos, mean_pred = calibration_curve(y_test, selected_proba, n_bins=5, strategy="quantile")

plt.figure(figsize=(6, 6))
plt.plot(mean_pred, frac_pos, marker="o", label=selected_model_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction positive")
plt.title("Calibration Curve — Selected Model")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_calibration_best_model_test.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 14. Quality Control Checklist
# ============================================================

qc_rows = []

def qc(status, item, detail):
    qc_rows.append({"qc_item": item, "status": status, "detail": detail})

qc("PASS" if NB04_DIR.exists() else "FAIL",
   "Notebook 04 output folder found",
   str(NB04_DIR))

qc("PASS" if all(verification_df["exists"]) else "FAIL",
   "Required input files found",
   f"Missing: {missing if missing else 'None'}")

qc("PASS" if len(X_train) == len(y_train) and len(X_test) == len(y_test) else "FAIL",
   "Feature/target row counts match",
   f"Train X/y: {X_train.shape[0]}/{len(y_train)}; Test X/y: {X_test.shape[0]}/{len(y_test)}")

qc("PASS" if y_train.nunique() == 2 and y_test.nunique() == 2 else "FAIL",
   "Both classes present in train and test",
   f"Train classes: {sorted(y_train.unique())}; Test classes: {sorted(y_test.unique())}")

qc("PASS" if X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0 else "FAIL",
   "No missing values in processed matrices",
   f"Train missing={int(X_train.isna().sum().sum())}; Test missing={int(X_test.isna().sum().sum())}")

qc("PASS",
   "Cross-validation performed within training set",
   "5-fold stratified cross-validation using training data only")

qc("PASS",
   "Threshold selected without test leakage",
   "Youden threshold derived from out-of-fold training predictions only")

qc("PASS" if len(test_perf) > 0 else "FAIL",
   "Held-out test evaluation completed",
   f"Models evaluated: {test_perf['model'].nunique() if len(test_perf)>0 else 0}")

qc("PASS" if not sens_cv_perf.empty and not sens_test_perf.empty else "WARN",
   "Sensitivity analysis excluding baseline_NP3TOT",
   "Completed" if not sens_cv_perf.empty and not sens_test_perf.empty else "Skipped or unavailable")

qc("PASS",
   "No preprocessing refit in Notebook 05",
   "Notebook 05 uses processed matrices from Notebook 04 and does not fit imputation/scaling/encoding.")

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(OUT_DIR / "11_quality_control_checklist.csv", index=False)
display(qc_df)

In [ ]:
# ============================================================
# 15. Summary report
# ============================================================

selected_test_row = test_perf[
    (test_perf["model"] == selected_model_name) &
    (test_perf["evaluation"] == "test_training_derived_youden_threshold")
].copy()

if len(selected_test_row) == 1:
    selected_test_metrics = selected_test_row.iloc[0].to_dict()
else:
    selected_test_metrics = {}

report_lines = []
report_lines.append("Notebook 05 — Baseline Machine Learning Models and Internal Validation")
report_lines.append("=" * 80)
report_lines.append("")
report_lines.append("Input source:")
report_lines.append(str(NB04_DIR))
report_lines.append("")
report_lines.append("Dataset:")
report_lines.append(f"- Train n: {len(y_train)}")
report_lines.append(f"- Test n: {len(y_test)}")
report_lines.append(f"- Processed predictors: {X_train.shape[1]}")
report_lines.append(f"- Train positive n: {int(y_train.sum())} ({y_train.mean()*100:.2f}%)")
report_lines.append(f"- Test positive n: {int(y_test.sum())} ({y_test.mean()*100:.2f}%)")
report_lines.append("")
report_lines.append("Selected primary model:")
report_lines.append(f"- {selected_model_name}")
report_lines.append(f"- Selection threshold: {selected_threshold:.6f}")
report_lines.append("- Selection rule: highest training CV ROC-AUC, then PR-AUC, then balanced accuracy.")
report_lines.append("")
if selected_test_metrics:
    report_lines.append("Held-out test performance for selected model using training-derived threshold:")
    for k in ["roc_auc", "pr_auc_average_precision", "balanced_accuracy", "sensitivity_recall", "specificity", "precision", "f1", "brier_score"]:
        report_lines.append(f"- {k}: {selected_test_metrics.get(k, np.nan):.4f}")
report_lines.append("")
report_lines.append("Sensitivity analysis:")
report_lines.append("- Excluding baseline_NP3TOT was performed if sensitivity feature names were available.")
report_lines.append("")
report_lines.append("Methodological notes:")
report_lines.append("- Test-set outcomes were not used to fit preprocessing, tune thresholds, or rank models.")
report_lines.append("- These are internal-validation baseline models, not final externally validated clinical tools.")
report_lines.append("- Calibration and external validation should be assessed before clinical interpretation.")
report_lines.append("")
report_lines.append("Generated files:")
for p in sorted(OUT_DIR.glob("*")):
    report_lines.append(f"- {p.name}")
report_lines.append("")
report_lines.append("Output folder:")
report_lines.append(str(OUT_DIR))

report_text = "\n".join(report_lines)
with open(OUT_DIR / "12_notebook_05_summary_report.txt", "w") as f:
    f.write(report_text)

print(report_text)

## Scientific Interpretation

Use the generated CSV files to interpret the results in this order:

1. `03_cv_performance_primary_models.csv`  
   Determine how models performed using training cross-validation.

2. `05_model_ranking.csv`  
   Confirm which model was selected without using the test set.

3. `04_test_performance_primary_models.csv`  
   Evaluate held-out test performance after model selection.

4. `09_sensitivity_test_performance_no_baseline_NP3TOT.csv`  
   Check whether predictive performance remains acceptable after removing baseline motor severity.

5. `10_feature_importance_best_model_training_only.csv`  
   Interpret the selected model cautiously. Feature importance is exploratory and should not be presented as causal evidence.

Do not move to final manuscript claims before reviewing calibration, sensitivity analysis, and potential external validation.

## Quality Control Checklist

Notebook 05 is complete only if:

- All required Notebook 04 files were found.
- Train/test row counts match target vectors.
- Both outcome classes are present in train and test sets.
- No missing values remain after preprocessing.
- Thresholds were selected using training cross-validation only.
- Held-out test evaluation was performed once after model selection.
- Sensitivity analysis excluding `baseline_NP3TOT` was generated.
- No external validation claim is made at this stage.